<a href="https://colab.research.google.com/github/ahmetyigitarslan435-eng/smart_car_rental_optimization/blob/main/RentManagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#Algorithm Project
import numpy as np
import pandas as pd
from datetime import datetime

def calculate_dynamic_price(start_day, end_day, base_rate, vehicle_segment):
    """
    Calculates the net dynamic rental price based on rental duration,
    vehicle segment, and high-demand pricing rules.
    """
    duration = end_day - start_day

    # 1. Vehicle Segment Multipliers
    segment_multipliers = {
        "Economy": 1.0,
        "Comfort": 1.5,
        "Luxury": 2.5
    }
    multiplier = segment_multipliers.get(vehicle_segment, 1.0)

    # Initial base calculation
    total_price = duration * base_rate * multiplier

    # 2. Dynamic Pricing Rules
    # Rule A: Long-term rental discount (10% off for 5 or more days)
    if duration >= 5:
        total_price *= 0.90

    # Rule B: Peak demand surcharge (20% extra between Day 3 and Day 7)
    if start_day >= 3 and start_day <= 7:
        total_price *= 1.20

    return round(total_price, 2)

def prepare_and_validate_dataset(raw_requests):
    """
    Validates rental request data, parses DD.MM.YYYY dates, calculates integer offsets
    relative to the earliest date, and returns a clean Pandas DataFrame.
    """
    valid_requests = []
    parsed_dates = []

    # First pass: Parse and validate date formats
    for req in raw_requests:
        try:
            start_dt = datetime.strptime(req["start_date"], "%d.%m.%Y")
            end_dt = datetime.strptime(req["end_date"], "%d.%m.%Y")
            parsed_dates.append((req, start_dt, end_dt))
        except ValueError:
            print(f"EDGE CASE WARNING: Request ID {req['id']} ignored! Invalid date format. Must be DD.MM.YYYY.")
            continue

    if not parsed_dates:
        return pd.DataFrame()

    # Find the earliest start date in the dataset to act as Epoch (Day 0)
    earliest_date = min(start_dt for _, start_dt, _ in parsed_dates)

    for req, start_dt, end_dt in parsed_dates:
        start_day = (start_dt - earliest_date).days
        end_day = (end_dt - earliest_date).days

        # Edge case 1: Invalid Data Range Check (End day must be strictly after start day)
        if end_day <= start_day:
            print(f"EDGE CASE WARNING: Request ID {req['id']} ignored! End date must be greater than start date.")
            continue
        # Edge case 2: Invalid price check (Base rate must be positive)
        if req["base_rate"] <= 0:
            print(f"EDGE CASE WARNING: Request ID {req['id']} ignored! Base rate must be positive.")
            continue

        # Calculate dynamic price using calculated integer offsets
        net_revenue = calculate_dynamic_price(
            start_day,
            end_day,
            req["base_rate"],
            req["vehicle_segment"]
        )

        valid_requests.append({
            "id": req["id"],
            "segment": req["vehicle_segment"],
            "start_date": req["start_date"],
            "end_date": req["end_date"],
            "start_day": start_day,
            "end_day": end_day,
            "duration": end_day - start_day,
            "net_revenue": net_revenue
        })

    # Convert clean dataset to Pandas Dataframe
    df = pd.DataFrame(valid_requests)
    return df

def get_latest_non_overlapping(jobs, index):
    """
    Finds the latest non-overlapping job that finishes before or at the start
    of job[index] using Binary Search.
    Time Complexity: O(log n)
    """
    low = 0
    high = index - 1
    while low <= high:
        mid = (low + high) // 2
        if jobs[mid]['end_day'] <= jobs[index]['start_day']:
            if mid + 1 < index and jobs[mid + 1]['end_day'] <= jobs[index]['start_day']:
                low = mid + 1
            else:
                return mid
        else:
            high = mid - 1
    return -1

def weighted_interval_scheduling(jobs):
    """
    Dynamic Programming (Weighted Interval Scheduling) Algorithm.
    Finds the global optimum subset of non-overlapping rental requests
    that maximizes total net revenue.
    """
    # Sort jobs by end_day in ascending order
    sorted_jobs = sorted(jobs, key=lambda x: x['end_day'])
    n = len(sorted_jobs)
    if n == 0:
        return 0, []

    dp = [0] * n
    selected_jobs_list = [[] for _ in range(n)]

    # Base case: first job
    dp[0] = sorted_jobs[0]['net_revenue']
    selected_jobs_list[0] = [sorted_jobs[0]]

    for i in range(1, n):
        # Option A: Include current job i
        incl_profit = sorted_jobs[i]['net_revenue']
        incl_list = [sorted_jobs[i]]

        # Find the latest non-overlapping job via binary search
        l = get_latest_non_overlapping(sorted_jobs, i)
        if l != -1:
            incl_profit += dp[l]
            incl_list = selected_jobs_list[l] + [sorted_jobs[i]]

        # Option B: Exclude current job i
        excl_profit = dp[i - 1]
        excl_list = selected_jobs_list[i - 1]

        # Choose the option that yields higher total revenue
        if incl_profit > excl_profit:
            dp[i] = incl_profit
            selected_jobs_list[i] = incl_list
        else:
            dp[i] = excl_profit
            selected_jobs_list[i] = excl_list

    return dp[n - 1], selected_jobs_list[n - 1]

def greedy_scheduling(jobs):
    """
    Greedy Algorithm (Earliest Finish Time heuristic).
    Sorts requests by earliest end_day and greedily picks non-overlapping jobs.
    Used as a baseline to demonstrate why Greedy can get stuck in a local optimum.
    """
    sorted_jobs = sorted(jobs, key=lambda x: x['end_day'])
    selected = []
    total_revenue = 0
    last_end_day = None

    for job in sorted_jobs:
        if last_end_day is None or job['start_day'] >= last_end_day:
            selected.append(job)
            total_revenue += job['net_revenue']
            last_end_day = job['end_day']

    return total_revenue, selected

def solve_rentals(df):
    """
    Main solver function integrating Member 1's validated DataFrame.
    Runs both Dynamic Programming and Greedy algorithms and compares results.
    """
    jobs = df.to_dict('records')

    dp_revenue, dp_selected = weighted_interval_scheduling(jobs)
    greedy_revenue, greedy_selected = greedy_scheduling(jobs)

    print("\n" + "=" * 55)
    print(" MEMBER 2: RENTAL REVENUE OPTIMIZATION COMPARISON ")
    print("=" * 55)
    print(f"Dynamic Programming (Global Optimum) Total Revenue : {dp_revenue}")
    print(f"Selected Request IDs (DP)                          : {[item['id'] for item in dp_selected]}")
    print("-" * 55)
    print(f"Greedy Algorithm (Local Optimum) Total Revenue     : {greedy_revenue}")
    print(f"Selected Request IDs (Greedy)                      : {[item['id'] for item in greedy_selected]}")
    print("=" * 55)

    return {
        "dp_revenue": dp_revenue,
        "dp_selected": dp_selected,
        "greedy_revenue": greedy_revenue,
        "greedy_selected": greedy_selected
    }

# Optimized DP with end-of-run selection reconstruction.
# Return the highest total revenue and the selected compatible requests.
def weighted_interval_scheduling_optimized(jobs):
    # Create a new list sorted by end day, leaving the input order intact.
    sorted_jobs = sorted(jobs, key=lambda job: job["end_day"])
    # Count the valid requests to process.
    n = len(sorted_jobs)
    # dp[k] is the best revenue for the first k requests. dp[0] is the empty plan.
    dp = [0] * (n + 1)
    # Store the last compatible index for each request.
    previous = [-1] * n
    # Record whether the best plan at each step includes the current request.
    take = [False] * n

    # enumerate provides the sorted index and the request record.
    for i, job in enumerate(sorted_jobs):
        # Find the earlier plan that can be combined with the current request.
        previous[i] = get_latest_non_overlapping(sorted_jobs, i)
        # Start the take total with the current request's own revenue.
        include = job["net_revenue"]
        # Add the best compatible earlier plan if one exists.
        if previous[i] != -1:
            # Add one because dp is indexed by the number of requests, not their zero-based index.
            include += dp[previous[i] + 1]
        # The first request is the base case. Later ties keep the previous plan.
        take[i] = i == 0 or include > dp[i]
        # Write the take total or carry forward the previous optimum.
        dp[i + 1] = include if take[i] else dp[i]

    selected = []
    # Begin tracing from the last request in sorted order.
    i = n - 1
    # Trace decisions until the index moves before the start of the list.
    while i >= 0:
        # Check whether the current request was taken.
        if take[i]:
            # Add the selected request to the result.
            selected.append(sorted_jobs[i])
            # Jump over intervening records to the last compatible index.
            i = previous[i]
        else:
            # If the request was skipped, continue one step back.
            i -= 1
    # Reverse the list because the requests were collected from last to first.
    selected.reverse()
    # Return the optimal revenue and the reservations that produce it.
    return dp[n], selected

from itertools import combinations
from math import isclose
from random import Random
from statistics import median
from time import perf_counter

def verify_algorithms():
    rng = Random(42)
    for size in range(9):
        for _ in range(10):
            jobs = []
            for i in range(size):
                start = rng.randrange(8)
                jobs.append(dict(id=i, start_day=start,
                                 end_day=start + rng.randrange(1, 5),
                                 net_revenue=rng.randrange(21)))
            optimum = 0
            for count in range(1, size + 1):
                for subset in combinations(jobs, count):
                    ordered = sorted(subset, key=lambda job: job["end_day"])
                    if all(a["end_day"] <= b["start_day"] for a, b in zip(ordered, ordered[1:])):
                        optimum = max(optimum, sum(job["net_revenue"] for job in ordered))
            result = weighted_interval_scheduling_optimized(jobs)
            assert result == weighted_interval_scheduling(jobs)
            assert result[0] == optimum
            assert greedy_scheduling(jobs)[0] <= optimum

verify_algorithms()
print("DP results matched exhaustive search on small randomized inputs.")

def verify_plan(result):
    revenue, selected = result
    assert all(a["end_day"] <= b["start_day"] for a, b in zip(selected, selected[1:]))
    assert isclose(revenue, sum(job["net_revenue"] for job in selected))

cases = {
    "Same-day turnover": ([(0, 2, 4), (2, 4, 6)], 10),
    "Fully overlapping requests": ([(0, 4, 3), (0, 4, 9), (0, 4, 7)], 9),
    "Earliest finish earns less": ([(0, 2, 4), (2, 4, 4), (0, 4, 10)], 10),
    "Single request": ([(1, 2, 5)], 5),
}
for label, (intervals, expected) in cases.items():
    jobs = [
        dict(id=i, start_day=start, end_day=end, net_revenue=revenue)
        for i, (start, end, revenue) in enumerate(intervals)
    ]
    dp = weighted_interval_scheduling_optimized(jobs)
    greedy = greedy_scheduling(jobs)
    verify_plan(dp)
    verify_plan(greedy)
    assert dp[0] == expected
    assert dp == weighted_interval_scheduling(jobs)
    print(f"{label} | Greedy: {greedy[0]}, DP: {dp[0]}")

def benchmark():
    results = []
    methods = {
        "Greedy": greedy_scheduling,
        "Original DP": weighted_interval_scheduling,
        "Optimized DP": weighted_interval_scheduling_optimized,
    }
    for size in (10, 10_000):
        # For benchmark, we simulate start_date and end_date using dummy date representations
        # so prepare_and_validate_dataset can compute the exact corresponding day offsets size-wise
        raw = [
            dict(id=i, start_date="01.01.2026", end_date="02.01.2026" if (i % 2 == 0) else "03.01.2026",
                 base_rate=100, vehicle_segment="Economy")
            for i in range(size)
        ]
        jobs_df = prepare_and_validate_dataset(raw)
        jobs = jobs_df.to_dict("records")
        expected = sum(job["net_revenue"] for job in jobs)
        selected_ids = [job["id"] for job in jobs]

        if size == 10_000:
            print(f"--- 10,000 Requests Test Result -> Max Revenue: {expected:,.2f} | Selected Count: {len(selected_ids)} ---")

        for label, algorithm in methods.items():
            timings = []
            for _ in range(3):
                start = perf_counter()
                revenue, selected = algorithm(jobs)
                timings.append(perf_counter() - start)
                assert isclose(revenue, expected)
                assert [job["id"] for job in selected] == selected_ids
            results.append(dict(requests=size, method=label,
                                milliseconds=median(timings) * 1000))
    return results

import matplotlib.pyplot as plt

# --- MUTUALLY EXCLUSIVE INTERACTIVE MENU ---
print("\n--- SELECT EXECUTION MODE ---")
print("1: Option 1 (Manual Live Input with DD.MM.YYYY Dates)")
print("2: Option 2 (10,000 Requests Scalability & Benchmark)")
try:
    choice = input("Enter option (1 or 2): ").strip()
except Exception:
    choice = "2"  # Fallback for non-interactive execution

if choice == "1":
    print("\n--- Option 1: Manual Live Input Mode ---")
    user_requests = []
    request_id = 1
    while True:
        print(f"\nEnter details for Request ID {request_id} (or leave segment blank to finish): ")
        segment = input("Vehicle Segment (Economy, Comfort, Luxury): ").strip()
        if not segment:
            break
        try:
            start_date_str = input("Start Date (DD.MM.YYYY, e.g. 15.11.2026): ").strip()
            end_date_str = input("End Date (DD.MM.YYYY, e.g. 20.11.2026): ").strip()
            base_rate = float(input("Base Rate (positive number): "))
            user_requests.append({
                "id": request_id,
                "vehicle_segment": segment,
                "start_date": start_date_str,
                "end_date": end_date_str,
                "base_rate": base_rate
            })
            request_id += 1
        except ValueError:
            print("Invalid input types. Please re-enter this request.")

    if len(user_requests) > 0:
        df_custom = prepare_and_validate_dataset(user_requests)
        if not df_custom.empty:
            print("\n--- Processed & Priced Rental Dataset ---")
            print(df_custom)
            solve_rentals(df_custom)
        else:
            print("No valid requests were recorded.")
    else:
        print("No requests entered. Exiting manual mode.")

else:
    print("\n--- Option 2: 10,000 Requests Scalability & Benchmark Mode ---")
    measurements = pd.DataFrame(benchmark())
    timing_table = measurements.pivot(index="requests", columns="method", values="milliseconds")
    print(timing_table.round(4).to_string())
    speedup = timing_table["Original DP"] / timing_table["Optimized DP"]
    print("\nOriginal DP runtime / optimized DP runtime")
    print(speedup.round(2).to_string())

    ax = timing_table.plot.bar(logy=True, rot=0, figsize=(9, 4))
    ax.set_xlabel("Number of requests")
    ax.set_ylabel("Median runtime (ms, log scale)")
    ax.set_title("Algorithm runtimes for 10 and 10,000 requests")
    ax.legend(title=None)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

DP results matched exhaustive search on small randomized inputs.
Same-day turnover | Greedy: 10, DP: 10
Fully overlapping requests | Greedy: 3, DP: 9
Earliest finish earns less | Greedy: 8, DP: 10
Single request | Greedy: 5, DP: 5

--- SELECT EXECUTION MODE ---
1: Option 1 (Manual Live Input with DD.MM.YYYY Dates)
2: Option 2 (10,000 Requests Scalability & Benchmark)
Enter option (1 or 2): 1

--- Option 1: Manual Live Input Mode ---

Enter details for Request ID 1 (or leave segment blank to finish): 
Vehicle Segment (Economy, Comfort, Luxury): Economy
Start Date (DD.MM.YYYY, e.g. 15.11.2026): 15.11.2026
End Date (DD.MM.YYYY, e.g. 20.11.2026): 18.11.2026
Base Rate (positive number): 500

Enter details for Request ID 2 (or leave segment blank to finish): 
Vehicle Segment (Economy, Comfort, Luxury): Luxury
Start Date (DD.MM.YYYY, e.g. 15.11.2026): 16.11.2026
End Date (DD.MM.YYYY, e.g. 20.11.2026): 22.11.2026
Base Rate (positive number): 1500

Enter details for Request ID 3 (or leave segme